### Data Quality Assessment

### Objective
Assess the integrity, completeness, consistency, and validity of the historical stock-price panel before exploratory data analysis and model development.

The data set contains the selected stocks from 'panel_filtered_80'.

In [1]:
import pandas as pd
import numpy as np

In [4]:
# loading the data set
df = pd.read_csv('penny_stocks_research/eligible_stocks.csv')
# standardise date format
df['date'] = pd.to_datetime(df['date'], errors ='coerce')
# sort observations
df = df.sort_values(['ticker','date']).reset_index(drop=True)

In [8]:
print('Dataset shape:', df.shape)
print('Number of stocks:', df['ticker'].nunique())
print('Date range', df['date'].min(), 'to', df['date'].max())
df.dtypes

Dataset shape: (64607, 12)
Number of stocks: 49
Date range 2020-01-02 00:00:00 to 2025-12-31 00:00:00


date             datetime64[ns]
Open                    float64
High                    float64
Low                     float64
Close                   float64
Adj Close               float64
Volume                    int64
Dividends               float64
Stock Splits            float64
Repaired?                  bool
ticker                   object
dollar_volume           float64
dtype: object

### 1. Missing values

Identify missing values by column and calculate their percenage of the dataset. Missing values will be investigated before deciding whether to remove or impute any observations.

In [9]:
missing_report = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': (df.isna().mean() * 100).round(4),
    'data_type':df.dtypes.astype(str)
})

In [12]:
missing_report.sort_values('missing_count', ascending=False)

,missing_count,missing_percentage,data_type
date,0,0.0,datetime64[ns]
Open,0,0.0,float64
High,0,0.0,float64
Low,0,0.0,float64
Close,0,0.0,float64
Adj Close,0,0.0,float64
Volume,0,0.0,int64
Dividends,0,0.0,float64
Stock Splits,0,0.0,float64
Repaired?,0,0.0,bool


### 2. Duplicate Records

Check for exact duplicate rows and duplicate ticker-date combinations. Each ticker should have no more than one observation per date in the daily panel.

In [13]:
exact_duplicates = df.duplicated().sum()

In [15]:
duplicate_ticker_dates = df.duplicated(
    subset=['ticker', 'date'], keep=False
).sum()

In [17]:
print('Exact duplicated rows', exact_duplicates)
print('Row involved in duplicated ticker-date pairs:', duplicate_ticker_dates)

if duplicate_ticker_dates >0:
    diplay(df.loc[df.duplicated(subset=['ticker','date'],keep=False).sort_values(['ticker','date'])])

Exact duplicated rows 0
Row involved in duplicated ticker-date pairs: 0


### 3. Price and Volume Validity

Identify non-positive prices, negative trading volume, and OHLC inconsistences. These checks flag suspicious records for investigation.

In [18]:
price_columns = ['Open','High','Low','Close']
# converting into numeric features
for col in price_columns + ['Volume']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        

In [20]:
# non-positive prices
invalid_prices = (
    df[price_columns].le(0).any(axis=1)
    if all(c in df.columns for c in price_columns)
    else pd.Series(False, index=df.index)
)

In [21]:
# Negative volume
negative_volume = (
    df["Volume"] < 0
    if "Volume" in df.columns
    else pd.Series(False, index=df.index)
)

In [22]:
# OHLC consistency: High should be at least all other OHLC values;
# Low should be no greater than all other OHLC values.
ohlc_columns_present = all(c in df.columns for c in price_columns)

if ohlc_columns_present:
    invalid_ohlc = (
        (df["High"] < df[["Open", "Low", "Close"]].max(axis=1))
        | (df["Low"] > df[["Open", "High", "Close"]].min(axis=1))
    )
else:
    invalid_ohlc = pd.Series(False, index=df.index)

In [23]:
print("Rows with non-positive OHLC prices:", invalid_prices.sum())
print("Rows with negative volume:", negative_volume.sum())
print("Rows with OHLC inconsistencies:", invalid_ohlc.sum())

display(df.loc[invalid_prices | negative_volume | invalid_ohlc].head(20))

Rows with non-positive OHLC prices: 0
Rows with negative volume: 0
Rows with OHLC inconsistencies: 0


,date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Repaired?,ticker,dollar_volume
